# Práctica: clasificando reseñas de compradores

En `reseñas.csv` tienes 20 reseñas de un e-commerce, cada una con su etiqueta de sentimiento (`positiva` o `negativa`). Es un dataset chico a propósito: no se trata de lograr el mejor accuracy posible, sino de que construyas el pipeline completo con tus propias manos y entiendas qué hace cada paso.
 
Antes de escribir código, leé las 20 reseñas del CSV a mano. Fijate qué palabras aparecen repetidas en las positivas y cuáles en las negativas. Esa intuición te va a servir después para saber si el modelo está aprendiendo algo razonable o si está fallando por algo que vos ya podías anticipar.

## Ejercicio 1: cargar y explorar
 
Cargá el CSV con Pandas. Contá cuántas reseñas hay de cada clase (`positiva` / `negativa`). Esto no es un paso decorativo: si las clases estuvieran muy desbalanceadas (por ejemplo 18 positivas y 2 negativas), un modelo podría lograr un accuracy alto simplemente prediciendo siempre "positiva", sin haber aprendido nada útil. Antes de entrenar cualquier modelo de clasificación, siempre chequeá el balance de clases.

In [ ]:
import pandas as pd

df=pd.read_csv('reseñas.csv')

pd.set_option('display.max_colwidth', None)

df


,texto,sentimiento
0,el producto llegó en perfecto estado y antes de lo esperado,positiva
1,"pésima experiencia, el paquete llegó abierto y faltaban piezas",negativa
2,"muy buena relación precio calidad, lo volvería a comprar",positiva
3,"el vendedor nunca respondió mis mensajes, mala atención",negativa
4,"superó mis expectativas, calidad excelente",positiva
5,"se rompió a la semana de uso, no lo recomiendo",negativa
6,"envío rapidísimo, tal cual la descripción",positiva
7,la caja llegó totalmente destruida y el producto dañado,negativa
8,"funciona perfecto, instalación sencilla y buen material",positiva
9,tardó un mes en llegar y encima vino incompleto,negativa


In [4]:
df['sentimiento'].value_counts()

sentimiento
positiva    10
negativa    10
Name: count, dtype: int64

## Ejercicio 2: tokenizar sin librerías
 
Escribí una función `tokenizar_simple(texto)` que reciba un string y devuelva una lista de palabras en minúscula, sin signos de puntuación, usando solo métodos nativos de Python (sin NLTK ni spaCy). Pista: puede que necesites la librería `string` y su lista de puntuación, además de `.split()`.
 
Después, aplicá `word_tokenize` de NLTK sobre la misma reseña y compará los resultados. ¿En qué casos tu función simple se equivoca o produce un resultado distinto al de NLTK? Escribí al menos un ejemplo concreto de una reseña del dataset donde la diferencia se note.

In [14]:
import string

def tokenizar_simple(texto):
    texto = texto.lower()

    for signo in string.punctuation:
        texto = texto.replace(signo, '')

    palabras = texto.split()

    return palabras

reseña = 'no funciona como se anuncia, una estafa total'

tokenizar_simple(reseña)

['no', 'funciona', 'como', 'se', 'anuncia', 'una', 'estafa', 'total']

In [25]:
import nltk
from nltk.tokenize import word_tokenize

word_tokenize(reseña)



['no', 'funciona', 'como', 'se', 'anuncia', ',', 'una', 'estafa', 'total']

## Ejercicio 3: el efecto de las stopwords
 
Tomá la reseña `"no lo recomiendo, una pérdida de dinero total"` (o equivalente del dataset) y sacale las stopwords en español con NLTK. Mirá el resultado.
 
Ahora respondé sin correr más código, solo pensando: si esta reseña fuera parte de un modelo de Bag of Words que sacó stopwords, y la palabra "no" desapareció, ¿qué información se perdió? Buscá en el dataset si hay alguna otra reseña donde sacar "no" cambiaría el sentido de la frase. Este ejercicio no tiene una única respuesta "correcta" en código: el objetivo es que argumentes cuándo sacar stopwords ayuda y cuándo perjudica.


1. La informacion que se perdio es la calificacion de la reseña, donde es negativa y se perdio parte de la negacion aunque sigan quedando partes negativas se modifica esa negacion quedando parte del significado original

2. Si, tenemos la reseña "el color no coincide para nada con la foto publicada", es un reseña negativa y si le sacamos la palabra "no" queda como una reseña positiva dando a entender que el color coincide con la foto publicada. 

In [29]:
from nltk.corpus import stopwords

texto = 'no lo recomiendo, una pérdida de dinero total'

token = word_tokenize(texto)

stopwords = (stopwords.words('spanish'))
token_filtrado = [t for t in token if t.lower() not in stopwords]

print(token)
print(token_filtrado)


['no', 'lo', 'recomiendo', ',', 'una', 'pérdida', 'de', 'dinero', 'total']
['recomiendo', ',', 'pérdida', 'dinero', 'total']


## Ejercicio 4: vectorizar con TF-IDF
 
Usando `TfidfVectorizer` de sklearn, vectorizá las 20 reseñas del dataset completo (no hace falta separar en train/test todavía). Imprimí el vocabulario completo que generó el vectorizador con `.get_feature_names_out()`.
 
Buscá en ese vocabulario las 5 palabras con el IDF más alto (es decir, las más "raras" o distintivas del corpus) y las 5 con el IDF más bajo. Podés acceder a esos valores con el atributo `.idf_` del vectorizador, en el mismo orden que el vocabulario. ¿Las palabras con IDF alto te parecen informativas sobre el sentimiento de la reseña donde aparecen?



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer()

Reseñas = vectorizador.fit_transform(df['texto'])

vocabulario = vectorizador.get_feature_names_out()
print(vocabulario)



['abierto' 'antes' 'anuncia' 'atención' 'avisarme' 'bien' 'buen' 'buena'
 'buenísima' 'caja' 'calidad' 'cancelaron' 'coincide' 'color' 'como'
 'compra' 'comprar' 'con' 'conforme' 'cual' 'cumplió' 'dañado' 'de' 'del'
 'descripción' 'destruida' 'diez' 'dudas' 'el' 'empaquetado' 'en' 'encima'
 'envío' 'es' 'esperado' 'estado' 'estafa' 'excelente' 'expectativas'
 'experiencia' 'faltaban' 'familia' 'foto' 'fotos' 'funciona' 'incompleto'
 'inferior' 'instalación' 'justo' 'la' 'las' 'llegar' 'llegó' 'lo' 'mala'
 'material' 'mensajes' 'mes' 'mi' 'mis' 'mostraban' 'muy' 'nada' 'no'
 'nunca' 'paquete' 'para' 'pedido' 'pedí' 'perfecto' 'piezas' 'precio'
 'producto' 'prometido' 'publicada' 'pésima' 'que' 'quedé' 'rapidísimo'
 'recomendé' 'recomiendo' 'reembolso' 'relación' 'resolvió' 'respondió'
 'rompió' 'se' 'semana' 'sencilla' 'servicio' 'sin' 'superó' 'tal' 'tardó'
 'terrible' 'todas' 'total' 'totalmente' 'un' 'una' 'uso' 'vendedor'
 'vino' 'volvería' 'ya']


In [ ]:
import pandas as pd

df_idf = pd.DataFrame({
    "palabra": vocabulario,
    "idf": vectorizador.idf_
})

df_idf.sort_values(by="idf", ascending=False).head(5)  #LOS 5 IDF MAS ALTOS

,palabra,idf
0,abierto,3.351375
1,antes,3.351375
2,anuncia,3.351375
4,avisarme,3.351375
5,bien,3.351375


In [ ]:
import pandas as pd

df_idf.sort_values(by="idf", ascending=True).head(5) #LOS 5 IDF MAS BAJOS

,palabra,idf
28,el,2.098612
49,la,2.098612
52,llegó,2.252763
53,lo,2.252763
22,de,2.435085
